In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

CREATE OR REPLACE TEMP VIEW order_payments_typed AS
SELECT
  order_id,
  TRY_CAST(payment_sequential AS INTEGER) AS payment_sequential,
  payment_type,
  TRY_CAST(payment_installments AS INTEGER) AS payment_installments,
  TRY_CAST(payment_value AS DECIMAL(10,2)) AS payment_value,
  _ingested_at
FROM workspace.bronze.order_payments
QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id, payment_sequential ORDER BY _ingested_at DESC) = 1;

CREATE OR REPLACE TABLE workspace.silver.order_payments AS
SELECT * FROM order_payments_typed
WHERE 1=1 
  AND order_id IS NOT NULL 
  AND payment_sequential IS NOT NULL 
  AND payment_type IS NOT NULL 
  AND payment_value IS NOT NULL  
  AND payment_value > 0
  AND payment_installments >= 1
  AND payment_installments IS NOT NULL
  AND payment_type != 'not_defined';

CREATE OR REPLACE TABLE workspace.silver.order_payments_quarantine AS
SELECT *
, CASE
    WHEN order_id IS NULL OR payment_sequential IS NULL
      THEN 'llave nula'
    WHEN payment_type IS NULL OR payment_type = 'not_defined'
      THEN 'tipo de pago no valido'
    WHEN payment_value IS NULL OR payment_value <= 0
      THEN 'monto pagado no valido'
    WHEN payment_installments IS NULL OR payment_installments < 1
      THEN 'plazos no validos'
  END AS motivo
FROM order_payments_typed
WHERE 1=1 AND (
  payment_value IS NULL OR payment_value <= 0
  OR payment_type IS NULL OR payment_type = 'not_defined'
  OR order_id IS NULL OR payment_sequential IS NULL
  OR payment_installments IS NULL OR payment_installments < 1);

In [0]:
%sql
Select 'bronze' as layer, count(*) from workspace.bronze.order_payments
UNION DISTINCT
Select 'silver' as layer, count(*) from workspace.silver.order_payments
UNION all
Select 'quarantine' as layer, count(*) from workspace.silver.order_payment_quarantine

In [0]:
%sql
SELECT motivo, COUNT(*) FROM workspace.silver.order_payments_quarantine GROUP BY motivo;